# 3PC Perf
调试3PC(ABY3)协议中的参数，对比性能

1. 128in, 8out
2. 256in, 256out
3. NLP precision

## (1) 初始化
### 1.1 加载模型和权重

In [1]:
import jax


from flax_rnn.helper import (
    load_from_cache, prefill, step_fn,
    sampler_baseline, sampler_greedy, sampler_min_p, sampler_top_k, sampler_legacy,
    generate_topk, generate_greedy, generate_minp,
    generate, generate_demo,
)

# base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
base_path = "/root/shared-nvme/hf_cache/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


In [2]:
generate_demo(model, params, tokenizer, "python is")

python is in a subdirectory, and the directory has a subdirectory, named, for instance,.pyx.

I have a project that uses Python, and I'd like to build a subdirectory to use the code from the project, without

### 1.2 初始化SPU模拟器

In [3]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "3pc.json",
    mode,
    bandwidth=100,
    latency=10
)

emulator.up()

FileNotFoundError: [Errno 2] No such file or directory: '/3pc.json'

## (2) 采样性能评估
### 2.1 初始化

In [ ]:
prompt = "python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
logits, _ = prefill(model, params, input_ids)
s_logits = emulator.seal(logits)

### 2.2 明文测试采样函数

- 基线：什么都不干，输入logits，返回0，作为基础通信量和时间的参考
- 贪心采样：取原始值最高的logit输出，无需softmax等计算，V-1次比较
- topk采样：取原始值最高的k个，在这k个中取样，O(kV)
- minp采样：找到最高概率pmax，设置最低概率pmin=minp*pmax，在概率pmin-pmax的logits中采样
- 老代码：跑出来非常糟糕结果的老采样代码

In [ ]:
next_id = sampler_baseline(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_top_k(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_min_p(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_legacy(logits)
print(prompt, tokenizer.decode(next_id), sep="")

### 2.3 spu测试
#### 2.3.1 baseline

In [ ]:
next_id = emulator.run(sampler_baseline)(s_logits)

In [ ]:
print(prompt, tokenizer.decode(next_id), sep="")

#### 2.3.2 greedy

In [ ]:
next_id = emulator.run(sampler_greedy)(s_logits)

In [ ]:
print(prompt, tokenizer.decode(next_id), sep="")

#### 2.3.3 top k

In [ ]:
next_id = emulator.run(sampler_top_k)(s_logits)

In [ ]:
print(prompt, tokenizer.decode(next_id), sep="")

#### 2.3.4 min p

In [ ]:
next_id = emulator.run(sampler_min_p)(s_logits)

In [ ]:
print(prompt, tokenizer.decode(next_id), sep="")

#### 2.3.5 legacy

In [ ]:
next_id = emulator.run(sampler_legacy)(s_logits)

In [ ]:
print(prompt, tokenizer.decode(next_id), sep="")

## (3) 验证推理性能
### 3.1 定义运行函数

In [ ]:
# 目的是规避model参数，给函数加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关
@jax.jit
def gen_spu_greedy(params, input_ids):
    return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快

@jax.jit
def gen_spu_topk(params, input_ids):
    return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢

@jax.jit
def gen_spu_minp(params, input_ids):
    return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

# @jax.jit
# def gen_spu(params, input_ids):
#     # return generate(model, params, input_ids, n_tokens_to_gen=3) # 老代码
#     # return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快
#     # return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢
#     return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

### 3.2 greedy

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_greedy)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_greedy)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.3 minp

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_minp)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_minp)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.4 topk

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_topk)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_topk)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

## (4) cleanup

In [ ]:
emulator.down()